# Scientific Workflows with ADEPT

Leverage ADEPT's scientific computing capabilities through Model Context Protocol (MCP) tools.

## Security Architecture

All API requests route through `nginx_proxy` (TLS + JWT authentication).
JupyterLab is isolated on `frontend_net` — MCP tools are invoked indirectly via the
chat/completions API, not by direct MCP server access.

## Available Scientific Tools

- **BLAST**: Sequence alignment and homology search
- **UniProt**: Protein database queries
- **PubChem**: Chemical compound information
- **AlphaFold**: Protein structure prediction
- **Nextflow**: Bioinformatics pipeline execution

## Docker Configuration
- JupyterLab service: `deployment/local/docker-compose/docker-compose.core.yaml`
- Rebuild after config changes: `make rebuild-jupyterlab`

In [ ]:
import os
import json
import httpx
import asyncio
import openai

# Load configuration from setup notebooks
config = json.load(open("/tmp/adept_config.json"))
auth = json.load(open("/tmp/adept_auth.json"))

API_BASE_URL = config.get('api_base_url', os.getenv('ADEPT_API_BASE_URL', 'https://nginx_proxy/v1'))

# Initialize OpenAI client — routes through nginx_proxy
client = openai.OpenAI(
    base_url=API_BASE_URL,
    api_key=auth['access_token'],
    http_client=httpx.Client(verify=False)
)

print("Scientific workflow client initialized")
print(f"   API: {API_BASE_URL}")
print(f"   Routing: nginx_proxy (TLS + JWT)")
print(f"   MCP tools: Invoked via chat completions (not direct access)")

## Example 1: Protein Analysis with BLAST

In [ ]:
# Protein sequence analysis via chat API (MCP tools invoked by orchestration layer)
protein_sequence = "MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTFSYGVQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITLGMDELYK"

response = client.chat.completions.create(
    model="agentic-framework:scientific-agent-v1",
    messages=[{
        "role": "user",
        "content": f"Analyze this protein sequence using BLAST: {protein_sequence[:50]}... Find similar proteins and their functions."
    }],
    max_tokens=2000
)

print("BLAST Analysis Results:")
print(response.choices[0].message.content)

## Example 2: Chemical Compound Research

In [ ]:
# Chemical compound research via PubChem MCP tool
def research_compound(compound_name: str):
    response = client.chat.completions.create(
        model="agentic-framework:scientific-agent-v1",
        messages=[{
            "role": "user",
            "content": f"Find information about {compound_name} using PubChem. Include molecular formula, weight, and biological activities."
        }],
        max_tokens=1500
    )
    print(f"\n{compound_name} Research:")
    print(response.choices[0].message.content)

research_compound("aspirin")
research_compound("caffeine")